Part 1: Supervised Fine-tuning and DARE

Prasanna Paithankar (21CS30065)

HuggingFace Hub Links:
- https://huggingface.co/PrasannaPaithankar/qwen2.5-1.5b-medical-sft-lora
- https://huggingface.co/PrasannaPaithankar/qwen2.5-1.5b-medical-sft-dare  

In [ ]:
import datetime
import json
import math
import subprocess
import time

import matplotlib

matplotlib.use("Agg")
from pathlib import Path
from typing import Dict

import evaluate
import matplotlib.pyplot as plt
import nltk
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi, login
from peft import (
    AutoPeftModelForCausalLM,
    LoraConfig,
    TaskType,
    get_peft_model,
)
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

import json

with open("config.json", "r") as f:
    CFG = json.load(f)

BASE_OUT = Path("./outputs")
LOG_DIR = BASE_OUT / "logs"
PLOT_DIR = BASE_OUT / "plots"
SFT_DIR = BASE_OUT / "model_sft_lora"
DARE_DIR = BASE_OUT / "model_sft_dare"
CKPT_DIR = BASE_OUT / "checkpoints" / "sft"

In [ ]:
login(token="")

logged in to HuggingFace as 'PrasannaPaithankar'


In [ ]:
raw_ds = load_dataset("medalpaca/medical_meadow_medqa", split="train")

README.md: 0.00B [00:00, ?B/s]

medical_meadow_medqa.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

In [ ]:
n = len(raw_ds)
n_train = int(0.60 * n)
n_val = int(0.20 * n)

ds_train = raw_ds.select(range(0, n_train))
ds_val = raw_ds.select(range(n_train, n_train + n_val))
ds_test = raw_ds.select(range(n_train + n_val, n))

if CFG["train_samples"]:
    ds_train = ds_train.select(range(min(CFG["train_samples"], len(ds_train))))
if CFG["val_samples"]:
    ds_val = ds_val.select(range(min(CFG["val_samples"], len(ds_val))))
if CFG["test_samples"]:
    ds_test = ds_test.select(range(min(CFG["test_samples"], len(ds_test))))

In [ ]:
SYSTEM_PROMPT = (
    "You are a knowledgeable and concise medical assistant. "
    "Answer the following medical question accurately and briefly."
)


def build_prompt(example: Dict) -> str:
    instruction = example.get("instruction", "").strip()
    inp = example.get("input", "").strip()
    output = example.get("output", "").strip()

    question = f"{instruction}\n{inp}".strip() if inp else instruction

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n{output}<|im_end|>"
    )
    return prompt


print(build_prompt(ds_train[0]))

<|im_start|>system
You are a knowledgeable and concise medical assistant. Answer the following medical question accurately and briefly.<|im_end|>
<|im_start|>user
Please answer with one of the option in the bracket
Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?? 
{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Doxycycline', 'E': 'Nitrofurantoin'},<|im_end|>
<|im_start|>assistant
E: Nitrofurantoin<|im_end|>


### SFT with LoRA

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
def tokenize(example):
    text = build_prompt(example)
    tokens = tokenizer(
        text,
        max_length=CFG["max_seq_len"],
        truncation=True,
        padding=False,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


print("tokenising train / val sets ...")
tok_train = ds_train.map(tokenize, remove_columns=ds_train.column_names, num_proc=2)
tok_val = ds_val.map(tokenize, remove_columns=ds_val.column_names, num_proc=2)
print(f"tokenised  ->  train: {len(tok_train)}  |  val: {len(tok_val)}")

tokenising train / val sets ...


Map (num_proc=2):   0%|          | 0/6106 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2035 [00:00<?, ? examples/s]

tokenised  ->  train: 6106  |  val: 2035


In [ ]:
print(f"loading base model: {BASE_MODEL} ...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()
print("base model loaded  |  gradient checkpointing enabled")

loading base model: Qwen/Qwen2.5-1.5B-Instruct ...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

base model loaded  |  gradient checkpointing enabled


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["lora_modules"],
    bias="none",
    inference_mode=False,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [18]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

In [19]:
class TimeEstimateCallback(TrainerCallback):
    """prints per-step progress bar with elapsed time and eta."""

    def __init__(self, total_steps: int):
        self.total_steps = total_steps
        self.start_time = None
        self._pbar = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self._pbar = tqdm(
            total=self.total_steps,
            desc="training",
            unit="step",
            dynamic_ncols=True,
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or self._pbar is None:
            return
        step = state.global_step
        if step == 0:
            return
        elapsed = time.time() - self.start_time
        frac_done = step / self.total_steps
        eta_s = elapsed / frac_done - elapsed if frac_done > 0 else 0
        loss = logs.get("loss", float("nan"))
        self._pbar.n = step
        self._pbar.set_postfix(
            {
                "loss": f"{loss:.4f}",
                "elapsed": str(datetime.timedelta(seconds=int(elapsed))),
                "eta": str(datetime.timedelta(seconds=int(eta_s))),
            }
        )
        self._pbar.refresh()

    def on_train_end(self, args, state, control, **kwargs):
        if self._pbar:
            self._pbar.close()
        elapsed = time.time() - self.start_time
        print(f"training complete in {str(datetime.timedelta(seconds=int(elapsed)))}")

In [20]:
steps_per_epoch = math.ceil(
    len(tok_train) / (CFG["per_device_train_bs"] * CFG["gradient_accum"])
)
total_steps = steps_per_epoch * CFG["num_epochs"]
print(f"steps per epoch: {steps_per_epoch}  |  total steps: {total_steps}")

steps per epoch: 382  |  total steps: 1146


In [ ]:
training_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=CFG["num_epochs"],
    per_device_train_batch_size=CFG["per_device_train_bs"],
    per_device_eval_batch_size=CFG["per_device_eval_bs"],
    gradient_accumulation_steps=CFG["gradient_accum"],
    learning_rate=CFG["learning_rate"],
    warmup_ratio=CFG["warmup_ratio"],
    weight_decay=CFG["weight_decay"],
    fp16=True,
    bf16=False,
    lr_scheduler_type="cosine",
    logging_dir=str(LOG_DIR / "trainer_logs"),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=2,
    group_by_length=True,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=CFG["early_stop_patience"]),
        TimeEstimateCallback(total_steps=total_steps),
    ],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [22]:
print("\n" + "=" * 60)
print("  starting sft training ...")
print("=" * 60)
train_result = trainer.train()

trainer.save_model(str(CKPT_DIR / "best"))
tokenizer.save_pretrained(str(CKPT_DIR / "best"))

train_metrics = train_result.metrics
trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)
print(f"\ntraining metrics: {train_metrics}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.



  starting sft training ...


training:   0%|          | 0/1146 [00:00<?, ?step/s]

Epoch,Training Loss,Validation Loss
1,1.072027,1.087026
2,0.994008,1.061409
3,0.923672,1.072032


training complete in 1:21:08
***** train metrics *****
  epoch                    =        3.0
  total_flos               = 38212845GF
  train_loss               =     1.0309
  train_runtime            = 1:21:08.70
  train_samples_per_second =      3.762
  train_steps_per_second   =      0.235

training metrics: {'train_runtime': 4868.7073, 'train_samples_per_second': 3.762, 'train_steps_per_second': 0.235, 'total_flos': 4.103073015003341e+16, 'train_loss': 1.0308617892273642, 'epoch': 3.0}


### Training Loss Curve

In [23]:
history = trainer.state.log_history
train_steps = [h["step"] for h in history if "loss" in h]
train_losses = [h["loss"] for h in history if "loss" in h]
eval_steps = [h["step"] for h in history if "eval_loss" in h]
eval_losses = [h["eval_loss"] for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_steps, train_losses, label="train loss", linewidth=1.5, color="#2196F3")
if eval_losses:
    ax.plot(
        eval_steps,
        eval_losses,
        label="val loss",
    )
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title("SFT Training Loss Curve")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
loss_plot_path = PLOT_DIR / "sft_loss_curve.png"
plt.savefig(loss_plot_path, dpi=150)
plt.show()
print(f"loss curve saved -> {loss_plot_path}")

loss curve saved -> outputs/plots/sft_loss_curve.png


In [24]:
del model
torch.cuda.empty_cache()

print("loading adapter + merging into full model ...")
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    str(CKPT_DIR / "best"),
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(str(SFT_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(SFT_DIR))
print(f"model_sft_lora saved -> {SFT_DIR}")

del merged_model
torch.cuda.empty_cache()

loading adapter + merging into full model ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model_sft_lora saved -> outputs/model_sft_lora


### Hyperparameter Summary

In [25]:
HYPERPARAMS = {
    "base_model": BASE_MODEL,
    "lora_r": CFG["lora_r"],
    "lora_alpha": CFG["lora_alpha"],
    "lora_dropout": CFG["lora_dropout"],
    "lora_modules": CFG["lora_modules"],
    "learning_rate": CFG["learning_rate"],
    "per_device_train_bs": CFG["per_device_train_bs"],
    "gradient_accum": CFG["gradient_accum"],
    "effective_batch_size": CFG["per_device_train_bs"] * CFG["gradient_accum"],
    "num_epochs": CFG["num_epochs"],
    "max_seq_len": CFG["max_seq_len"],
    "warmup_ratio": CFG["warmup_ratio"],
    "weight_decay": CFG["weight_decay"],
    "early_stop_patience": CFG["early_stop_patience"],
    "optimizer": "paged_adamw_8bit",
    "lr_scheduler": "cosine",
    "quantization": "fp16 + gradient checkpointing (no bnb on sm_60)",
}

print("\n" + "=" * 50)
print("  sft hyperparameters")
print("=" * 50)
for k, v in HYPERPARAMS.items():
    print(f"  {k:<28} {v}")

sft_log = {
    "hyperparameters": HYPERPARAMS,
    "train_metrics": {
        k: (v if isinstance(v, (int, float, str)) else str(v))
        for k, v in train_metrics.items()
    },
}
with open(LOG_DIR / "part1_sft_log.json", "w") as f:
    json.dump(sft_log, f, indent=2)
print(f"\nhyperparameters + metrics saved -> {LOG_DIR / 'part1_sft_log.json'}")


  sft hyperparameters
  base_model                   Qwen/Qwen2.5-1.5B-Instruct
  lora_r                       16
  lora_alpha                   32
  lora_dropout                 0.05
  lora_modules                 ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  learning_rate                0.0002
  per_device_train_bs          4
  gradient_accum               4
  effective_batch_size         16
  num_epochs                   3
  max_seq_len                  512
  warmup_ratio                 0.03
  weight_decay                 0.01
  early_stop_patience          2
  optimizer                    paged_adamw_8bit
  lr_scheduler                 cosine
  quantization                 fp16 + gradient checkpointing (no bnb on sm_60)

hyperparameters + metrics saved -> outputs/logs/part1_sft_log.json


In [26]:
if CFG["hf_upload"]:
    sft_repo_id = f"{CFG['hf_username']}/{CFG['sft_repo_name']}"
    print(f"pushing model_sft_lora -> {sft_repo_id} ...")
    try:
        api = HfApi()
        api.create_repo(
            repo_id=sft_repo_id, repo_type="model", exist_ok=True, private=False
        )
        api.upload_folder(
            folder_path=str(SFT_DIR),
            repo_id=sft_repo_id,
            repo_type="model",
            commit_message="add model_sft_lora: Qwen2.5-1.5B finetuned on medqa (LoRA merged)",
        )
        print(f"model_sft_lora uploaded -> https://huggingface.co/{sft_repo_id}")
    except Exception as e:
        print(f"warning: hf upload failed for model_sft_lora — {e}")
        print("  model weights are saved locally and can be uploaded manually.")
else:
    print("skipping hf upload (test mode).")

pushing model_sft_lora -> PrasannaPaithankar/qwen2.5-1.5b-medical-sft-lora ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

model_sft_lora uploaded -> https://huggingface.co/PrasannaPaithankar/qwen2.5-1.5b-medical-sft-lora


### DARE Application via mergekit

In [ ]:
def make_dare_yaml(base_model_path: str, sft_model_path: str, density: float) -> str:
    # density = 1 - p (fraction of parameters kept after dropout)
    # mergekit dare_linear implements the dare paper directly
    config = f"""
models:
  - model: {base_model_path}
    parameters:
      density: 1.0
      weight: 1.0
  - model: {sft_model_path}
    parameters:
      density: {density:.2f}
      weight: 1.0
merge_method: dare_linear
base_model: {base_model_path}
dtype: bfloat16
"""
    return config.strip()


def run_dare(base_path: str, sft_path: str, drop_p: float, out_dir: Path) -> Path:
    density = 1.0 - drop_p
    yaml_content = make_dare_yaml(base_path, sft_path, density)

    cfg_path = BASE_OUT / f"dare_p{drop_p:.1f}.yaml"
    cfg_path.write_text(yaml_content)

    dare_out = out_dir / f"dare_p{drop_p:.1f}"
    dare_out.mkdir(parents=True, exist_ok=True)

    print(
        f"running dare  |  p={drop_p}  |  density={density:.2f}  |  output -> {dare_out}"
    )

    cmd = [
        "mergekit-yaml",
        str(cfg_path),
        str(dare_out),
        "--copy-tokenizer",
        "--allow-crimes",
        "--trust-remote-code",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"mergekit-yaml failed:\n{result.stderr}")
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    return dare_out


def evaluate_model_on_val(
    model_path: str, ds: Dataset, n_samples: Optional[int] = None
) -> Dict:
    if n_samples:
        ds = ds.select(range(min(n_samples, len(ds))))

    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")
    bleu_metric = evaluate.load("bleu")

    print(f"  loading model from {model_path} for evaluation ...")
    eval_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    eval_tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    eval_tok.pad_token = eval_tok.eos_token
    eval_model.eval()

    preds, refs = [], []
    for ex in tqdm(ds, desc="  evaluating", leave=False):
        question = f"{ex.get('instruction', '')}\n{ex.get('input', '')}".strip()
        prompt_text = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{question}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = eval_tok(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=CFG["max_seq_len"],
        ).to(device)
        with torch.no_grad():
            out = eval_model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=1.0,
                pad_token_id=eval_tok.eos_token_id,
            )
        generated = eval_tok.decode(
            out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
        ).strip()
        reference = ex.get("output", "").strip()
        preds.append(generated)
        refs.append(reference)

    rouge_res = rouge_metric.compute(predictions=preds, references=refs)
    meteor_res = meteor_metric.compute(predictions=preds, references=refs)
    bleu_res = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])

    scores = {
        "rouge_l": round(rouge_res["rougeL"], 4),
        "meteor": round(meteor_res["meteor"], 4),
        "bleu": round(bleu_res["bleu"], 4),
    }

    del eval_model
    torch.cuda.empty_cache()
    return scores

### DARE Sweep

In [28]:
DARE_TEMP_DIR = BASE_OUT / "dare_candidates"
dare_results: Dict[float, Dict] = {}

for p in tqdm(CFG["dare_p_values"], desc="dare sweep"):
    t0 = time.time()
    dare_path = run_dare(
        base_path=BASE_MODEL,
        sft_path=str(SFT_DIR),
        drop_p=p,
        out_dir=DARE_TEMP_DIR,
    )
    merge_time = time.time() - t0
    print(f"  merge complete in {merge_time:.1f}s")

    eval_n = CFG["dare_eval_samples"]
    print(f"  evaluating on {'all' if eval_n is None else eval_n} val examples ...")
    scores = evaluate_model_on_val(str(dare_path), ds_val, n_samples=eval_n)
    dare_results[p] = {
        "scores": scores,
        "path": str(dare_path),
        "merge_time_s": merge_time,
    }
    print(
        f"  p={p}  ->  rouge-l: {scores['rouge_l']:.4f} | meteor: {scores['meteor']:.4f} | bleu: {scores['bleu']:.4f}"
    )

print("\n" + "=" * 55)
print("  dare sweep summary")
print("=" * 55)
for p, info in dare_results.items():
    s = info["scores"]
    print(
        f"  p={p}  ->  rouge-l: {s['rouge_l']} | meteor: {s['meteor']} | bleu: {s['bleu']}"
    )

best_p = max(dare_results, key=lambda p: dare_results[p]["scores"]["rouge_l"])
best_scores = dare_results[best_p]["scores"]
best_dare_src = dare_results[best_p]["path"]
print(f"\nbest drop rate: p={best_p}  (rouge-l={best_scores['rouge_l']})")

dare sweep:   0%|          | 0/4 [00:00<?, ?it/s]

running dare  |  p=0.1  |  density=0.90  |  output -> outputs/dare_candidates/dare_p0.1

  merge complete in 41.5s
  evaluating on all val examples ...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


  loading model from outputs/dare_candidates/dare_p0.1 for evaluation ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


  loading model from outputs/dare_candidates/dare_p0.7 for evaluation ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  evaluating:   0%|          | 0/2035 [00:00<?, ?it/s]

  p=0.7  ->  rouge-l: 0.4726 | meteor: 0.5178 | bleu: 0.4459

  dare sweep summary
  p=0.1  ->  rouge-l: 0.4742 | meteor: 0.5194 | bleu: 0.4441
  p=0.3  ->  rouge-l: 0.4796 | meteor: 0.5246 | bleu: 0.4495
  p=0.5  ->  rouge-l: 0.4765 | meteor: 0.521 | bleu: 0.4499
  p=0.7  ->  rouge-l: 0.4726 | meteor: 0.5178 | bleu: 0.4459

best drop rate: p=0.3  (rouge-l=0.4796)


In [29]:
ps = sorted(dare_results.keys())
rl = [dare_results[p]["scores"]["rouge_l"] for p in ps]
meteor = [dare_results[p]["scores"]["meteor"] for p in ps]
bleu_v = [dare_results[p]["scores"]["bleu"] for p in ps]

x = np.arange(len(ps))
width = 0.25

fig, ax = plt.subplots(figsize=(max(7, len(ps) * 2.5), 4))
bars1 = ax.bar(x - width, rl, width, label="ROUGE-L", color="#2196F3", alpha=0.85)
bars2 = ax.bar(x, meteor, width, label="METEOR", color="#4CAF50", alpha=0.85)
bars3 = ax.bar(x + width, bleu_v, width, label="BLEU", color="#FF9800", alpha=0.85)

for bars in (bars1, bars2, bars3):
    for bar in bars:
        ax.annotate(
            f"{bar.get_height():.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )

ax.set_xlabel("dare drop rate (p)")
ax.set_ylabel("score")
ax.set_title("DARE Sweep: Validation-Set Performance vs Drop Rate")
ax.set_xticks(x)
ax.set_xticklabels([f"p={p}" for p in ps])
ax.axvline(
    x=ps.index(best_p),
    color="red",
    linestyle="--",
    alpha=0.5,
    label=f"best: p={best_p}",
)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
dare_plot_path = PLOT_DIR / "dare_sweep.png"
plt.savefig(dare_plot_path, dpi=150)
plt.show()
print(f"dare sweep plot saved -> {dare_plot_path}")

dare sweep plot saved -> outputs/plots/dare_sweep.png


In [30]:
print(f"copying best dare model (p={best_p}) to {DARE_DIR} ...")
if DARE_DIR.exists():
    shutil.rmtree(DARE_DIR)
shutil.copytree(best_dare_src, str(DARE_DIR))
print(f"model_sft_dare saved -> {DARE_DIR}")

copying best dare model (p=0.3) to outputs/model_sft_dare ...
model_sft_dare saved -> outputs/model_sft_dare


In [31]:
if CFG["hf_upload"]:
    dare_repo_id = f"{CFG['hf_username']}/{CFG['dare_repo_name']}"
    print(f"pushing model_sft_dare -> {dare_repo_id} ...")
    try:
        api = HfApi()
        api.create_repo(
            repo_id=dare_repo_id, repo_type="model", exist_ok=True, private=False
        )
        api.upload_folder(
            folder_path=str(DARE_DIR),
            repo_id=dare_repo_id,
            repo_type="model",
            commit_message=(
                f"add model_sft_dare: dare(p={best_p}) applied on model_sft_lora"
            ),
        )
        print(f"model_sft_dare uploaded -> https://huggingface.co/{dare_repo_id}")
    except Exception as e:
        print(f"warning: hf upload failed for model_sft_dare — {e}")
        print("  model weights are saved locally and can be uploaded manually.")
else:
    print("skipping hf upload (test mode).")

pushing model_sft_dare -> PrasannaPaithankar/qwen2.5-1.5b-medical-sft-dare ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

model_sft_dare uploaded -> https://huggingface.co/PrasannaPaithankar/qwen2.5-1.5b-medical-sft-dare


In [ ]:
final_log = {
    "config_mode": CONFIG,
    "base_model": BASE_MODEL,
    "dataset": "medalpaca/medical_meadow_medqa",
    "split_sizes": {
        "train": len(ds_train),
        "val": len(ds_val),
        "test": len(ds_test),
    },
    "sft_hyperparameters": HYPERPARAMS,
    "sft_train_metrics": {
        k: (v if isinstance(v, (int, float, str)) else str(v))
        for k, v in train_metrics.items()
    },
    "dare_sweep": {
        str(p): {
            "rouge_l": info["scores"]["rouge_l"],
            "meteor": info["scores"]["meteor"],
            "bleu": info["scores"]["bleu"],
            "merge_time_s": round(info["merge_time_s"], 1),
        }
        for p, info in dare_results.items()
    },
    "best_dare_p": best_p,
    "best_dare_scores": best_scores,
    "hf_repos": {
        "model_sft_lora": f"https://huggingface.co/{CFG['hf_username']}/{CFG['sft_repo_name']}",
        "model_sft_dare": f"https://huggingface.co/{CFG['hf_username']}/{CFG['dare_repo_name']}",
    }
    if CFG["hf_upload"]
    else "upload_disabled",
    "plots": {
        "sft_loss_curve": str(loss_plot_path),
        "dare_sweep": str(dare_plot_path),
    },
    "timestamp": datetime.datetime.now().isoformat(),
}

log_path = LOG_DIR / "part1_final_log.json"
with open(log_path, "w") as f:
    json.dump(final_log, f, indent=2)

if CFG["hf_upload"]:
    print(f"\n  HuggingFace repos:")
    print(f"    https://huggingface.co/{CFG['hf_username']}/{CFG['sft_repo_name']}")
    print(f"    https://huggingface.co/{CFG['hf_username']}/{CFG['dare_repo_name']}")
print(f"{'─' * 60}")
print(f"\n  best dare drop rate: p={best_p}")
print(f"  best dare rouge-l:  {best_scores['rouge_l']}")
print(f"  best dare meteor:   {best_scores['meteor']}")
print(f"  best dare bleu:     {best_scores['bleu']}")

  HuggingFace repos:
    https://huggingface.co/PrasannaPaithankar/qwen2.5-1.5b-medical-sft-lora
    https://huggingface.co/PrasannaPaithankar/qwen2.5-1.5b-medical-sft-dare

  best dare drop rate: p=0.3
  best dare rouge-l:  0.4796
  best dare meteor:   0.5246
  best dare bleu:     0.4495
